In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
symlist = self.get_updated_symbol_list(age_ub=165000)

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_earning = self.count_days_from_earning_reports(df_quotes)
if df_earning.shape[0] == 0:
    d2e = {}
else:
    d2e = df_earning['earningDays'].to_dict()
print('Days to E:', d2e)
df_raw = self.build_option_df(symlist)
px.bar(self.check_data_age(df_raw), barmode='group', width=60*len(symlist), height=300).show()
df_put = self.select_options_by_type(df_raw, 'put')

In [ ]:
df_call = self.select_options_by_type(df_raw, 'call')
df_put = self.select_options_by_type(df_raw, 'put')
dfcp = self.concat_put_call_options(df_raw)

In [ ]:
df_call.shape, df_put.shape, dfcp.shape

In [ ]:
df_put.tail()

In [ ]:
df_call.tail()

In [ ]:
dfcp.tail()

In [ ]:
_df = self.calc_spread_stats(df_call)
px.bar(_df.sort_values(by='mean spread'), barmode='group', width=60*len(symlist))

In [ ]:
dfp = self.compute_all_time_decay_metrics_for_symbols(df_put, symlist, d2e, ignore_no_bid=True, exclude_0dte=True, oi_lb=100)
print('hdte_resid check:', dfp[(dfp.hdte_resid < dfp.resid) & (np.abs(dfp.hdte_resid - dfp.resid) >= 1e-6)].shape)

### Put options with no earning date on or before expiration date

In [ ]:
hdte_resid_ub = 0.75
spread_ub = 5
moneyness_ub = 0.98
premium_lb = 1
delta_lb = -0.15
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub) & (dfp.E.isna() |(dfp.E > dfp.dte))
_filter = _filter & (dfp.premium >= premium_lb) & (dfp.Delta >= delta_lb)
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
print(_dfp.shape)
_dfp.head(60)

In [ ]:
_df = dfcp[(dfcp.type=='P') & (dfcp.symbol=='SPY')]
for s in _df.strike.unique():
    print(s, self.get_theta_curve(dfcp, 'SPY', s, 'P').set_index('dte').to_dict())